In [39]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss

%store -r df
adf_result = adfuller(df['h-cleaned'].dropna(), regression='c', result_object=True)
kpss_result = kpss(df['h-cleaned'].dropna(), regression='c', result_object=True )
print('--------- ADF - test  -------------- ')
print(adf_result)
print('--------- KPSS - test  -------------- ')
print(kpss_result)

--------- ADF - test  -------------- 
ADFullerResult
ADF Statistic: -0.87122
P-value: 0.79730
Used Lag: 8
Nobs: 1507
Critical Values: {'1%': np.float64(-3.434696697851678), '5%': np.float64(-2.8634597924260303), '10%': np.float64(-2.567792072971565)}

--------- KPSS - test  -------------- 
KPSSResult
KPSS Statistic: 4.73369
P-value: 0.01000
Lags: 25
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}



In [40]:
# функции для работы с сырыми массивами (raw=True)
def get_adf_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return adfuller(x, regression="c", autolag="AIC", result_object=True).pvalue
    except:
        return np.nan

def get_kpss_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return kpss(x, regression="c", nlags="auto", result_object=True).pvalue
    except:
        return np.nan

def stationarity_rolling_window(df, column_name, min_pct=0.20, max_pct=0.70, step_pct=0.05, alpha_adf=0.05, alpha_kpss=0.05):
    series = df[column_name].dropna()
    total_len = len(series)

    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))
    window_sizes = range(min_window, max_window + 1, step)

    print(f"Анализ ряда длиной {total_len}")

    successful_windows = [] # для сводной таблицы по p-value тестов

    for window_size in window_sizes:
        adf_series = series.rolling(window=window_size).apply(get_adf_p, raw=True)
        kpss_series = series.rolling(window=window_size).apply(get_kpss_p, raw=True)

        # иаска для выполнений условий стационарности, применяемая к рядам
        valid_mask = (adf_series < alpha_adf) & (kpss_series >= alpha_kpss)

        # Сохраняем столбец ADF в итоговый датафрейм (только валидные значения, остальные NaN)
        df[f'ADF {window_size}'] = adf_series.where(valid_mask)
        df[f'KPSS {window_size}'] = kpss_series.where(valid_mask)

        # информация для таблицы results_df только по успешным точкам
        end_indices = series.index[valid_mask]
        for end_label in end_indices:
            # найти порядковый номер конечной точки в исходном ряду
            end_idx = series.index.get_loc(end_label)
            start_idx = end_idx - window_size + 1
            start_label = series.index[start_idx]

            successful_windows.append({
                "window_size": window_size,
                "window_pct": round(window_size / total_len, 2),
                "start_index": start_label,
                "end_index": end_label,
                "start_datetime": df.loc[start_label, "DATETIME"] if "DATETIME" in df.columns else start_label,
                "end_datetime": df.loc[end_label, "DATETIME"] if "DATETIME" in df.columns else end_label,
                "adf_p_value": adf_series.loc[end_label],
                "kpss_p_value": kpss_series.loc[end_label] })

    results_df = pd.DataFrame(successful_windows)

    # удаляем колонки ADF и KPSS со значениями nan, оставляем только те где выполняются словия и по ADF и по KPSS
    df = df.loc[:, ~(df.columns.str.startswith('ADF ') | df.columns.str.startswith('KPSS ')) | df.notna().any()]
    return results_df, df

In [41]:
r_window_min_pct = float(input(f'Enter MINIMAL % for rolling window in decimal format:'))
r_window_max_pct = float(input(f'Enter MAXIMUM % for rolling window in decimal format:'))
r_window_step_pct = float(input(f'Enter STEP % for rolling window in decimal format:'))
adf_p_user = float(input(f'Enter ADF p-value in decimal format:'))
kpss_p_user =float(input(f'Enter KPSS p-value in decimal format:'))
print(f'''Parameters:
        min rolling window: {r_window_min_pct * 100}%
        max rolling window: {r_window_max_pct* 100}%
        window step:        {r_window_step_pct * 100}%
        ADF p-value used:   {adf_p_user}
        KPSS p-value used:  {adf_p_user}''')

Parameters:
        min rolling window: 30.0%
        max rolling window: 70.0%
        window step:        3.0%
        ADF p-value used:   0.05
        KPSS p-value used:  0.05


In [42]:
import warnings
from statsmodels.tools.sm_exceptions import (InterpolationWarning, ValueWarning)
warnings.filterwarnings('ignore', category=InterpolationWarning)
warnings.filterwarnings('ignore', category=ValueWarning)

pvalue_res, df_aug = stationarity_rolling_window(df=df, column_name="h-cleaned", min_pct=r_window_min_pct, max_pct=r_window_max_pct, step_pct=r_window_step_pct, alpha_adf=adf_p_user, alpha_kpss=kpss_p_user)
df_aug.head()

Анализ ряда длиной 1516


,TICKER,DATETIME,CLOSE,h-cleaned,ADF 454,KPSS 454,ADF 499,KPSS 499,ADF 544,KPSS 544,...,ADF 679,KPSS 679,ADF 724,KPSS 724,ADF 769,KPSS 769,ADF 814,KPSS 814,ADF 859,KPSS 859
0,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:00:00,1343.0,1343.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:15:00,1334.0,1334.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:30:00,1330.0,1330.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:45:00,1330.0,1330.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SiZ6SiH7 [FUTSPREAD],2026-10-01 08:00:00,1325.0,1325.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
nedf=df_aug.dropna(subset=df_aug.columns[4:], how='all')
nedf

,TICKER,DATETIME,CLOSE,h-cleaned,ADF 454,KPSS 454,ADF 499,KPSS 499,ADF 544,KPSS 544,...,ADF 679,KPSS 679,ADF 724,KPSS 724,ADF 769,KPSS 769,ADF 814,KPSS 814,ADF 859,KPSS 859
504,SiZ6SiH7 [FUTSPREAD],2026-10-09 21:45:00,1249.0,1249.0,0.048401,0.072658,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
505,SiZ6SiH7 [FUTSPREAD],2026-10-09 22:00:00,1255.0,1255.0,0.048119,0.075326,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
506,SiZ6SiH7 [FUTSPREAD],2026-10-09 22:15:00,1255.0,1255.0,0.047057,0.078028,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507,SiZ6SiH7 [FUTSPREAD],2026-10-09 22:30:00,1255.0,1255.0,0.044446,0.080552,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
508,SiZ6SiH7 [FUTSPREAD],2026-10-09 22:45:00,1258.0,1258.0,0.026631,0.082795,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1482,SiZ6SiH7 [FUTSPREAD],2026-10-28 15:30:00,1431.0,1431.0,NaN,NaN,NaN,NaN,0.029520,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1483,SiZ6SiH7 [FUTSPREAD],2026-10-28 15:45:00,1417.0,1417.0,NaN,NaN,0.037908,0.100000,0.021692,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1484,SiZ6SiH7 [FUTSPREAD],2026-10-28 16:00:00,1425.0,1425.0,NaN,NaN,0.043883,0.096513,0.024639,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,SiZ6SiH7 [FUTSPREAD],2026-10-28 16:15:00,1430.0,1430.0,NaN,NaN,NaN,NaN,0.031143,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug['h-cleaned'], name=df_aug['TICKER'].iloc[1]), secondary_y=False)
# числовые суффиксы ADF
suffixes = sorted({match.group(1) for col in df_aug.columns if (match := re.fullmatch(r'ADF\s+(\d+)', col))}, key=int)  # подсказал ИИ

for suffix in suffixes:
    adf_col = f'ADF {suffix}'
    kpss_col = f'KPSS {suffix}'

    if kpss_col not in df_aug.columns:
        continue

    for col in [adf_col, kpss_col]:
        fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug[col], name=f'{col} rolling'), secondary_y=True)

fig.show()

In [45]:
# def adf_rolling(x):
#     return adfuller(x, regression='c', autolag="AIC", result_object=True).pvalue
# def kpss_rolling(x):
#     return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
# # window = 300
# # df['ADF roll'] = df['<CLOSE>'].rolling(window).apply(adf_rolling, raw=True)
# # df['KPSS roll'] = df['<CLOSE>'].rolling(window).apply(kpss_rolling, raw=True)
# # df.iloc[:61]

In [46]:
# def stationarity_results_to_df (df, min_pct=0.20, max_pct=0.70, step_pct=0.05):
#     total_len = len(df['CLOSE'])
#     min_window = int(min_pct * total_len)
#     max_window = int(max_pct * total_len)
#     step_window = max(1, int(step_pct * total_len))
#     window_sizes = range(min_window, max_window + 1, step_window)
#
#     for window in window_sizes:
#         df[f'ADF {window}'] = df['CLOSE'].rolling(window).apply(adf_rolling, raw=True)
#         df[f'KPSS {window}'] = df['CLOSE'].rolling(window).apply(kpss_rolling, raw=True)
#
#     return df

In [47]:
# min_pct = float(input('enter min % in decimal format'))
# max_pct = float(input('enter max bond for percentage in decimal format'))
# step_pct = float(input('enter step % in decimal format'))
#
# new_df = stationarity_results_to_df(df, min_pct, max_pct, step_pct)